In [ ]:
import pandas as pd
import os

os.listdir("Data")

In [ ]:
wheel_wear = pd.read_excel("Data/2023-06-13 - Wheel wear results.xlsx")

wheel_wear.head()

In [ ]:
wheel_wear.shape

In [ ]:
eq_file = pd.ExcelFile("Data/Eq_Con_Fit_Results.xlsm")

eq_file.sheet_names

In [ ]:
eq_con = pd.read_excel(
    "Data/Eq_Con_Fit_Results.xlsm",
    sheet_name="Eq_Con_Fit_Results"
)

eq_con.head()

In [ ]:
eq_con.shape

In [ ]:
# Remove the 7 KRWHL rows since they do not have matching Car/Axle data in the wheel wear file
eq_con_cars = eq_con[
    eq_con["Filename"].str.startswith("Car", na=False)
].copy()


eq_con_cars.shape

In [ ]:
# Extract the car number, axle number, and rail profile number from each filename
eq_con_cars["Car"] = eq_con_cars["Filename"].str.extract(r"Car(\d+)").astype(int)
eq_con_cars["Axle"] = eq_con_cars["Filename"].str.extract(r"Axle(\d+)").astype(int)
eq_con_cars["Rail_Profile"] = eq_con_cars["Filename"].str.extract(r"RProfile\s*(\d+)").astype(int)

# Check that the new columns were extracted correctly
eq_con_cars[["Filename", "Car", "Axle", "Rail_Profile"]].head(10)

In [ ]:
# Extract the car number, axle number, and left/right wheel side from each wheel wear filename
wheel_wear["Car"] = wheel_wear["Filename"].str.extract(r"Car(\d+)").astype(int)
wheel_wear["Axle"] = wheel_wear["Filename"].str.extract(r"Axle(\d+)").astype(int)
wheel_wear["Side"] = wheel_wear["Filename"].str.extract(r"Axle\d+([LR])")

# Check that the new columns were extracted correctly
wheel_wear[["Filename", "Car", "Axle", "Side"]].head(10)

In [ ]:
# Show all columns in the wheel wear dataset before reshaping the L/R wheel data
wheel_wear.columns.tolist()

In [ ]:
# Remove all blank Excel columns labeled "Unnamed" so only the actual wheel wear data remains
wheel_wear = wheel_wear.loc[
    :, ~wheel_wear.columns.str.startswith("Unnamed:")
]

# Check the remaining column names
wheel_wear.columns.tolist()

In [ ]:
wheel_wear.dtypes

In [ ]:
# Remove identifier/text columns so only the numeric wheel measurements remain
wheel_measurements = wheel_wear.drop(
    columns=[
        "Filename",
        "Axle Number",
        "Wheel Side",
        "Hollow wear"
    ]
)

# Reshape the data so each Car/Axle has separate left and right wheel measurements
wear_axle = wheel_measurements.pivot(
    index=["Car", "Axle"],
    columns="Side"
)

# Rename the left and right measurement columns
new_columns = []

for measurement, side in wear_axle.columns:
    new_name = measurement + "_" + side
    new_columns.append(new_name)

wear_axle.columns = new_columns

# Turn Car and Axle back into regular columns
wear_axle = wear_axle.reset_index()

# Check the size
wear_axle.shape

In [ ]:
# Check the number of rows and columns after combining the left and right wheels
print(wear_axle.shape)

# Show the column names to make sure each measurement has an L and R version
wear_axle.columns.tolist()

In [ ]:
# Merge the wheel measurements into the Eq Con data by matching the same Car and Axle
merged_data = eq_con_cars.merge(
    wear_axle,
    on=["Car", "Axle"],
    how="left",
    validate="many_to_one"
)

# Check the size of the merged dataset
merged_data.shape

In [ ]:
# Check that every Eq Con row received matching wheel wear data
missing_rows = merged_data["qR_L"].isna().sum()

print("Rows missing wheel wear data:", missing_rows)

In [ ]:
# List the four hunting-speed columns that can be used as our outcome variables
hunting_columns = [
    "Empty Truck from Acc HS [mph]",
    "Empty Axle from Lat Disp HS [mph]",
    "Loaded Truck from Acc HS [mph]",
    "Loaded Axle from Lat Disp HS [mph]"
]

# Check the data type of each hunting-speed column without displaying the actual values
merged_data[hunting_columns].dtypes

In [ ]:
# Classify each row as hunting if the empty-car hunting speed is below 80 mph
merged_data["Empty_Truck_Hunted"] = merged_data["Empty Truck from Acc HS [mph]"] < 80
merged_data["Empty_Axle_Hunted"] = merged_data["Empty Axle from Lat Disp HS [mph]"] < 80

# Count the hunting and non-hunting cases for each method
print(merged_data["Empty_Truck_Hunted"].value_counts())
print(merged_data["Empty_Axle_Hunted"].value_counts())

In [ ]:
# Save the completed merged dataset, will add eq truck data next
merged_data.to_csv("Data/merged_wheel_hunting_data.csv", index=False)

merged_data.columns.tolist()

In [ ]:
# Add the truck number so each axle can be matched with the correct truck-level data
merged_data["Truck"] = 1

merged_data.loc[merged_data["Axle"] >= 3, "Truck"] = 2

# Check how many rows belong to each truck
merged_data["Truck"].value_counts()

In [ ]:
# Load the truck-level results from the second sheet of the Eq Con Excel file
truck_eq = pd.read_excel(
    "Data/Eq_Con_Fit_Results.xlsm",
    sheet_name="Truck_Eq_Con_Res"
)

# Check the size of the truck dataset
truck_eq.shape

# Keep only the truck rows associated with Cars 1–14 and remove the KRWHL rows
truck_cars = truck_eq[
    truck_eq["Filename"].str.startswith("Car", na=False)
].copy()

# Check the size of the remaining truck dataset
truck_cars.shape

In [ ]:
# Extract the car, truck, and rail profile numbers so the truck data can match our existing merged dataset
truck_cars["Car"] = truck_cars["Filename"].str.extract(r"Car(\d+)").astype(int)
truck_cars["Truck"] = truck_cars["Filename"].str.extract(r"Truck(\d+)").astype(int)
truck_cars["Rail_Profile"] = truck_cars["Filename"].str.extract(r"RProfile\s*(\d+)").astype(int)

# Check that the identifiers were extracted correctly
truck_cars[["Filename", "Car", "Truck", "Rail_Profile"]].head(10)

In [ ]:
# Merge the truck-level results into the existing dataset using Car, Truck, and Rail Profile
final_data = merged_data.merge(
    truck_cars,
    on=["Car", "Truck", "Rail_Profile"],
    how="left",
    suffixes=("", "_Truck"),
    validate="many_to_one"
)

# Check the size of the final dataset
final_data.shape

In [ ]:
# Check that every row received matching truck-level data
missing_truck = final_data["Min Eq Con 3mm"].isna().sum()

print("Rows missing truck data:", missing_truck)

In [ ]:
# Save the final merged dataset into the Data folder
final_data.to_csv("Data/final_merged_data.csv", index=False)

**We combined the wheel wear, axle-level Eq Con, and truck-level Eq Con data into one dataset.
The Wheel Wear Results originally had 112 rows representing 14 cars, 4 axles per car, and a left and right wheel for each axle. We removed the blank Unnamed columns, created Car, Axle, and Side identifiers from the filenames, and reshaped the data so each axle had one row with separate left and right measurements. For example, qR became qR_L and qR_R. This resulted in 56 axle rows.**

**The Eq Con Fit Results contained 392 Car rows, representing 14 cars × 4 axles × 7 rail profiles. We created Car, Axle, and Rail_Profile identifiers from the filenames and merged the wheel wear data with it using Car and Axle. This connected the physical L/R wheel measurements with the Eq Con and hunting results for each axle under each rail profile.**

**We also created Empty_Truck_Hunted and Empty_Axle_Hunted. Based on the community partner's rule for these columns, a hunting speed below 80 mph means hunting occurred, while 80 mph means no hunting occurred.**

**The Truck Eq Con Results contained 196 rows, representing 14 cars × 2 trucks × 7 rail profiles. Truck 1 contains Axles 1 and 2, while Truck 2 contains Axles 3 and 4. We added a Truck identifier to our existing data, created Car, Truck, and Rail_Profile identifiers for the truck data, and merged it using those three identifiers. Truck columns that had the same names as existing columns received a _Truck suffix.**

**The final dataset has 392 rows and 94 columns. Each row represents a specific Car + Axle + Rail Profile and contains the left and right wheel wear measurements, axle-level Eq Con and hunting results, corresponding truck-level results, and hunting classifications. All 392 rows successfully matched with the wheel wear and truck data.**